In [1]:
%%configure -f
{
    "executorMemory": "1g",
    "executorCores": 1,
    "driverMemory": "1g",
    "driverCores": 1,
    "numExecutors": 1
}

StatementMeta(sparkpool, 50, -1, Finished, Available, Finished, False)

See https://go.microsoft.com/fwlink/?linkid=2170827


In [8]:
%run Utilities

StatementMeta(sparkpool, 25, 2, Finished, Available, Finished, False)

In [74]:
from pyspark.sql.functions import col,count,when,isnull, to_date, to_timestamp, trim, current_date, current_timestamp
from pyspark.sql.types import IntegerType, DoubleType
from datetime import datetime

StatementMeta(sparkpool, 25, 68, Finished, Available, Finished, False)

In [9]:
data = read_csv_df("abfss://bronze@adlstoragesiddharth.dfs.core.windows.net/Financial Transaction Fraud Dataset/FraudShield_Banking_Data.csv",infer_schema=True)

display(data)

StatementMeta(sparkpool, 25, 3, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 92352836-6c14-4fed-ba30-ff0a474bc095)

In [41]:
print(f"Total Records: {data.count()}")
print(f"Total columns {len(data.columns)}")
data.printSchema()

StatementMeta(sparkpool, 25, 35, Finished, Available, Finished, False)

Total Records: 50000
Total columns 25
root
 |-- Transaction_ID: double (nullable = true)
 |-- Customer_ID: double (nullable = true)
 |-- Transaction_Amount (in Million): double (nullable = true)
 |-- Transaction_Time: timestamp (nullable = true)
 |-- Transaction_Date: date (nullable = true)
 |-- Transaction_Type: string (nullable = true)
 |-- Merchant_ID: double (nullable = true)
 |-- Merchant_Category: string (nullable = true)
 |-- Transaction_Location: string (nullable = true)
 |-- Customer_Home_Location: string (nullable = true)
 |-- Distance_From_Home: double (nullable = true)
 |-- Device_ID: double (nullable = true)
 |-- IP_Address: string (nullable = true)
 |-- Card_Type: string (nullable = true)
 |-- Account_Balance (in Million): double (nullable = true)
 |-- Daily_Transaction_Count: double (nullable = true)
 |-- Weekly_Transaction_Count: double (nullable = true)
 |-- Avg_Transaction_Amount (in Million): double (nullable = true)
 |-- Max_Transaction_Last_24h (in Million): double

In [47]:
# 1. Check for duplicates

duplicate_count = spark.sql("""
    SELECT (COUNT(Transaction_ID)- COUNT(DISTINCT(Transaction_ID))) AS `Duplicate Records`
    FROM {df}""",df=data
)

display(duplicate_count)

StatementMeta(sparkpool, 25, 41, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, c182c85d-bff2-4f53-89d8-7c19af01a7be)

In [52]:
# Check null values

null_counts = data.select([count(when(isnull(col(cols)), cols)).alias(cols) for cols in data.columns])
display(null_counts)

StatementMeta(sparkpool, 25, 46, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 43436d40-146c-4f47-9433-f7fb14cb1658)

In [66]:
# Remove duplicates

data_clean = data.dropDuplicates(['Transaction_ID'])

StatementMeta(sparkpool, 25, 60, Finished, Available, Finished, False)

In [67]:
# Handle column names and trim spaces

data_clean = data_clean.select([col(cols).alias(cols.strip()) for cols in data_clean.columns])

StatementMeta(sparkpool, 25, 61, Finished, Available, Finished, False)

In [70]:
# Data Type conversions

data_clean = (data_clean
    .withColumn("Transaction_ID", col("Transaction_ID").cast(IntegerType()))
    .withColumn("Customer_ID" , col("Customer_ID").cast(IntegerType()))
    .withColumn("Merchant_ID", col("Merchant_ID").cast(IntegerType()))
    .withColumn("Device_ID", col("Device_ID").cast(IntegerType()))
    .withColumn("Transaction_ID", to_date(col("Transaction_Date"), "yyyy-MM-dd"))
)

StatementMeta(sparkpool, 25, 64, Finished, Available, Finished, False)

In [71]:
data_clean = data_clean.filter(
    (col("Transaction_ID").isNotNull()) &
    (col("Customer_ID").isNotNull()) &
    (col("Transaction_date").isNotNull())
)

StatementMeta(sparkpool, 25, 65, Finished, Available, Finished, False)

In [75]:
# Add metadata columns

data_clean = (data_clean.
    withColumn("load_date", current_date())
    .withColumn("load_timestamp", current_timestamp())
)

StatementMeta(sparkpool, 25, 69, Finished, Available, Finished, False)

In [76]:
print(f"Total Records: {data_clean.count()}")
display(data_clean.limit(10))

StatementMeta(sparkpool, 25, 70, Finished, Available, Finished, False)

Total Records: 48628


SynapseWidget(Synapse.DataFrame, 63edfe04-2639-4b10-aad2-649111c37985)

In [78]:
# Rename columns to remove special characters
data_clean = data_clean.select([
    col(c).alias(c.replace(" (in Million)", "").replace(" ", "_").replace("-", "_").lower()) 
    for c in data_clean.columns
])

# Write to silver layer
data_clean.write.format("delta").mode("overwrite").save(
    "abfss://silver@adlstoragesiddharth.dfs.core.windows.net/fraud_detection/clean_transactions"
)

StatementMeta(sparkpool, 25, 72, Finished, Available, Finished, False)

In [79]:
display(data_clean.limit(5))

StatementMeta(sparkpool, 25, 73, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, c1d7be19-d988-48f8-80a8-b4631aa0a110)